In [40]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import sklearn as skt
import xgboost as xgb
import json
from scipy.stats import trim_mean
import math
from pandas.api.types import ( is_numeric_dtype, is_categorical_dtype, is_object_dtype, is_datetime64_any_dtype )
from default_risk.scripts.cv_mlfow_integration import run_cv_tracked_mlflow
import default_risk.config as cfg
import os
import xgboost as xgb
from dotenv import load_dotenv
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import mlflow
import mlflow.xgboost
import dtale


bureau_df = pd.read_parquet(cfg.CLEANS_DIR / "bureau.train-cleaned.parquet")

load_dotenv()
experiment_name = os.getenv("MLFLOW_EXPERIMENT_NAME", "default_experiment")
mlflow.set_experiment(experiment_name)
mlflow.xgboost.autolog(log_models=True)

In [47]:
bureau_df.sort_values(["id_curr", "days_credit"],inplace=True,ascending=False)
last_three = bureau_df.groupby("id_curr").head(3)
last_three = last_three.copy()
last_three["loan_order"] = last_three.groupby("id_curr").cumcount() + 1
last_three_columns = last_three.pivot(index="id_curr", columns="loan_order")
last_three_columns.columns =[f"{col}_prev_{rank}" for col, rank in last_three_columns.columns]


dtale.show(last_three)



In [ ]:
# In the case that there are multiple modes, we use the most resent 
def get_first_mode(x):
    mode_series = x.mode()
    return mode_series.iloc[0] if not mode_series.empty else pd.NA

bureau_aggregattted = bureau_df.groupby("id_curr").agg({
    "id_curr": ["count"],
    "flag_have_credit_day_overdue": ["count"],
    "credit_type": [get_first_mode],
    "amt_credit_sum_limit": ["max"],
    "cnt_credit_prolong": ["max", "min", "mean"],
    "amt_credit_sum": ["max", "min", "mean"],
    
})

dtale.show(bureau_aggregattted)


In [46]:
combined_rows = pd.concat([bureau_aggregattted, last_three_columns], axis=0, ignore_index=True)
dtale.show(combined_rows)
